In [ ]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [ ]:

import os
import json
import numpy as np
from bayes_opt import BayesianOptimization
from config import Config

def evenly_binary_sequence(count: int, n: int) -> np.ndarray:
    count = int(np.clip(count, 0, n))
    seq = np.zeros(n, dtype=np.int32)
    if count == 0:
        return seq
    if count == n:
        seq[:] = 1
        return seq

    acc = 0
    for i in range(n):
        acc += count
        if acc >= n:
            seq[i] = 1
            acc -= n
    return seq

def counts_to_binary_action_plan(
    counts_block: np.ndarray,
    num_od: int,
    total_step: int,
    block_steps: int,
) -> np.ndarray:
    n_blocks = int(np.ceil(total_step / block_steps))
    action_plan = np.zeros((total_step, num_od), dtype=np.int32)

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s

        for od in range(num_od):
            c = int(np.rint(counts_block[b, od]))
            c = int(np.clip(c, 0, n_local))
            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)

    return action_plan

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(worker_idx=worker_idx, total_step=action_plan.shape[0], seed=seed)
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def sequential_bayes_optimize_counts_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=2,
    n_iter=8,
    seed=42,
    eval_seed=None,
    verbose=2,
):
\
\
\
\
       
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    fixed_counts = np.zeros((0, num_od), dtype=np.float32)
    block_logs = []
    eval_count = {"n": 0}

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s

        if b == 0:
            prefix_reward = 0.0
        else:
            prefix_plan = counts_to_binary_action_plan(
                counts_block=fixed_counts,
                num_od=num_od,
                total_step=s,
                block_steps=block_steps,
            )
            prefix_reward, _ = run_episode_with_plan(
                build_env_fn=build_env_fn,
                worker_idx=worker_idx,
                action_plan=prefix_plan,
                seed=seed + eval_count["n"],
            )
            eval_count["n"] += 1

        pbounds = {f"od{od}": (0.0, float(n_local)) for od in range(num_od)}

        def objective(**kwargs):
            cur_counts = np.array(
                [np.clip(kwargs[f"od{od}"], 0.0, float(n_local)) for od in range(num_od)],
                dtype=np.float32
            )[None, :]

            candidate_counts = np.vstack([fixed_counts, cur_counts])
            candidate_plan = counts_to_binary_action_plan(
                counts_block=candidate_counts,
                num_od=num_od,
                total_step=e,
                block_steps=block_steps
            )

            total_reward_to_e, _ = run_episode_with_plan(
                build_env_fn=build_env_fn,
                worker_idx=worker_idx,
                action_plan=candidate_plan,
                seed=seed + eval_count["n"],
            )
            eval_count["n"] += 1

            return float(total_reward_to_e - prefix_reward)

        bo = BayesianOptimization(
            f=objective,
            pbounds=pbounds,
            random_state=seed + b,
            verbose=verbose,
            allow_duplicate_points=True,
        )
        bo.maximize(init_points=init_points, n_iter=n_iter)

        best_cur = np.array(
            [np.clip(bo.max["params"][f"od{od}"], 0.0, float(n_local)) for od in range(num_od)],
            dtype=np.float32
        )[None, :]

        fixed_counts = np.vstack([fixed_counts, best_cur])

        committed_plan = counts_to_binary_action_plan(
            counts_block=fixed_counts,
            num_od=num_od,
            total_step=e,
            block_steps=block_steps,
        )
        committed_reward_to_e, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=committed_plan,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1

        block_logs.append({
            "block": int(b),
            "step_start": int(s),
            "step_end": int(e),
            "n_local": int(n_local),
            "best_counts": best_cur.flatten().tolist(),
            "best_block_objective": float(bo.max["target"]),
            "committed_reward_to_block_end": float(committed_reward_to_e),
        })

        print(
            f"[block {b}] steps {s}~{e} | best_counts={best_cur.flatten().tolist()} | "
            f"block_obj={bo.max['target']:.6f} | committed_R={committed_reward_to_e:.6f}"
        )

    best_actions = counts_to_binary_action_plan(
        counts_block=fixed_counts,
        num_od=num_od,
        total_step=total_step,
        block_steps=block_steps,
    )

    replay_seed = seed if eval_seed is None else eval_seed
    final_reward, final_traj = run_episode_with_plan(
        build_env_fn=build_env_fn,
        worker_idx=worker_idx,
        action_plan=best_actions,
        seed=replay_seed,
    )

    return best_actions, fixed_counts, final_reward, final_traj, block_logs

In [ ]:
import os
import sys
import json
import time
import tempfile
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from config import Config
from trial_timing import reset_trial_times, append_trial_time

TRIALS = list(range(5))
MAX_WORKERS = len(TRIALS)
_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
os.makedirs(Config.RESULT_DIR, exist_ok=True)
reset_trial_times(_trial_result_dir)

_runner_source = "import os\nimport sys\nimport json\nimport time\nfrom pathlib import Path\n_scripts_dir = Path(os.environ['DODE_SCRIPTS_DIR']).resolve()\nos.chdir(_scripts_dir)\nif str(_scripts_dir) not in sys.path:\n    sys.path.insert(0, str(_scripts_dir))\ntrial = int(sys.argv[1])\n\nimport os\nimport shutil\nimport numpy as np\nfrom config import Config\nfrom env import MySumoEnv\n\n_NEEDED_WORKER_FILES = (\"detectors.add.xml\", \"network.net.xml\", \"run.sumocfg\")\n\ndef prepare_worker_dir(worker_idx=0):\n    if hasattr(Config, \"ensure_dirs\"):\n        Config.ensure_dirs()\n    else:\n        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)\n    src = Config.BASE_WORK_DIR\n    dst = Config.worker_dir(worker_idx)\n    os.makedirs(dst, exist_ok=True)\n    for name in _NEEDED_WORKER_FILES:\n        src_path = os.path.join(src, name)\n        dst_path = os.path.join(dst, name)\n        if not os.path.exists(src_path):\n            raise FileNotFoundError(f\"No utils file: {src_path}\")\n        shutil.copy2(src_path, dst_path)\n    os.makedirs(os.path.join(dst, \"dump\"), exist_ok=True)\n    return dst\n\ndef build_env(worker_idx=0, total_step=None, seed=42):\n    rl_dir = prepare_worker_dir(worker_idx)\n    answer_path = Config.answer_path_for(worker_idx)\n    sumocfg_path = os.path.join(rl_dir, \"run.sumocfg\")\n\n    if not os.path.exists(rl_dir):\n        raise FileNotFoundError(f\"No directory: {rl_dir}\")\n    if not os.path.exists(sumocfg_path):\n        raise FileNotFoundError(f\"No run.sumocfg: {sumocfg_path}\")\n    if not os.path.exists(answer_path):\n        raise FileNotFoundError(f\"No answer.csv: {answer_path}\")\n\n    if total_step is None:\n        total_step = Config.TOTAL_STEP\n\n    env = MySumoEnv(\n        rl_dir=rl_dir,\n        sumo_binary=Config.SUMO_BINARY,\n        origin_list=Config.ORIGIN_LIST,\n        destination_list=Config.DESTINATION_LIST,\n        input_interval=Config.INPUT_INTERVAL,\n        detector_interval=Config.DETECTOR_INTERVAL,\n        num_OD=Config.NUM_OD,\n        state_dim=1,\n        answer_dir=answer_path,\n        total_step=int(total_step),\n    )\n\n    obs, info = env.reset(seed=seed)\n    return env, obs, info\n\n\nimport os\nimport json\nimport numpy as np\nfrom bayes_opt import BayesianOptimization\nfrom config import Config\n\ndef evenly_binary_sequence(count: int, n: int) -> np.ndarray:\n    count = int(np.clip(count, 0, n))\n    seq = np.zeros(n, dtype=np.int32)\n    if count == 0:\n        return seq\n    if count == n:\n        seq[:] = 1\n        return seq\n\n    acc = 0\n    for i in range(n):\n        acc += count\n        if acc >= n:\n            seq[i] = 1\n            acc -= n\n    return seq\n\ndef counts_to_binary_action_plan(\n    counts_block: np.ndarray,\n    num_od: int,\n    total_step: int,\n    block_steps: int,\n) -> np.ndarray:\n    n_blocks = int(np.ceil(total_step / block_steps))\n    action_plan = np.zeros((total_step, num_od), dtype=np.int32)\n\n    for b in range(n_blocks):\n        s = b * block_steps\n        e = min((b + 1) * block_steps, total_step)\n        n_local = e - s\n\n        for od in range(num_od):\n            c = int(np.rint(counts_block[b, od]))\n            c = int(np.clip(c, 0, n_local))\n            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)\n\n    return action_plan\n\ndef run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):\n    env, obs, info = build_env_fn(worker_idx=worker_idx, total_step=action_plan.shape[0], seed=seed)\n    total_reward = 0.0\n    last_info = {}\n\n    try:\n        for t in range(action_plan.shape[0]):\n            obs, reward, terminated, truncated, info = env.step(action_plan[t])\n            total_reward += float(reward)\n            last_info = info\n            if terminated or truncated:\n                break\n    finally:\n        env.close()\n\n    trajectory = last_info.get(\"trajectory\", [])\n    return float(total_reward), trajectory\n\ndef sequential_bayes_optimize_counts_300s(\n    build_env_fn,\n    worker_idx,\n    num_od,\n    total_step,\n    input_interval,\n    detector_interval,\n    init_points=2,\n    n_iter=8,\n    seed=42,\n    eval_seed=None,\n    verbose=2,\n):\n    \"\"\"\n    Sequential optimization in 300-second (=60-step) blocks\n    - when optimizing block b, simulate through the end of block b\n    - previous blocks are fixed to their committed best actions\n    \"\"\"\n    block_steps = detector_interval // input_interval\n    n_blocks = int(np.ceil(total_step / block_steps))\n\n    fixed_counts = np.zeros((0, num_od), dtype=np.float32)\n    block_logs = []\n    eval_count = {\"n\": 0}\n\n    for b in range(n_blocks):\n        s = b * block_steps\n        e = min((b + 1) * block_steps, total_step)\n        n_local = e - s\n\n        if b == 0:\n            prefix_reward = 0.0\n        else:\n            prefix_plan = counts_to_binary_action_plan(\n                counts_block=fixed_counts,\n                num_od=num_od,\n                total_step=s,\n                block_steps=block_steps,\n            )\n            prefix_reward, _ = run_episode_with_plan(\n                build_env_fn=build_env_fn,\n                worker_idx=worker_idx,\n                action_plan=prefix_plan,\n                seed=seed + eval_count[\"n\"],\n            )\n            eval_count[\"n\"] += 1\n\n        pbounds = {f\"od{od}\": (0.0, float(n_local)) for od in range(num_od)}\n\n        def objective(**kwargs):\n            cur_counts = np.array(\n                [np.clip(kwargs[f\"od{od}\"], 0.0, float(n_local)) for od in range(num_od)],\n                dtype=np.float32\n            )[None, :]\n\n            candidate_counts = np.vstack([fixed_counts, cur_counts])\n            candidate_plan = counts_to_binary_action_plan(\n                counts_block=candidate_counts,\n                num_od=num_od,\n                total_step=e,\n                block_steps=block_steps\n            )\n\n            total_reward_to_e, _ = run_episode_with_plan(\n                build_env_fn=build_env_fn,\n                worker_idx=worker_idx,\n                action_plan=candidate_plan,\n                seed=seed + eval_count[\"n\"],\n            )\n            eval_count[\"n\"] += 1\n\n            return float(total_reward_to_e - prefix_reward)\n\n        bo = BayesianOptimization(\n            f=objective,\n            pbounds=pbounds,\n            random_state=seed + b,\n            verbose=verbose,\n            allow_duplicate_points=True,\n        )\n        bo.maximize(init_points=init_points, n_iter=n_iter)\n\n        best_cur = np.array(\n            [np.clip(bo.max[\"params\"][f\"od{od}\"], 0.0, float(n_local)) for od in range(num_od)],\n            dtype=np.float32\n        )[None, :]\n\n        fixed_counts = np.vstack([fixed_counts, best_cur])\n\n        committed_plan = counts_to_binary_action_plan(\n            counts_block=fixed_counts,\n            num_od=num_od,\n            total_step=e,\n            block_steps=block_steps,\n        )\n        committed_reward_to_e, _ = run_episode_with_plan(\n            build_env_fn=build_env_fn,\n            worker_idx=worker_idx,\n            action_plan=committed_plan,\n            seed=seed + eval_count[\"n\"],\n        )\n        eval_count[\"n\"] += 1\n\n        block_logs.append({\n            \"block\": int(b),\n            \"step_start\": int(s),\n            \"step_end\": int(e),\n            \"n_local\": int(n_local),\n            \"best_counts\": best_cur.flatten().tolist(),\n            \"best_block_objective\": float(bo.max[\"target\"]),\n            \"committed_reward_to_block_end\": float(committed_reward_to_e),\n        })\n\n        print(\n            f\"[block {b}] steps {s}~{e} | best_counts={best_cur.flatten().tolist()} | \"\n            f\"block_obj={bo.max['target']:.6f} | committed_R={committed_reward_to_e:.6f}\"\n        )\n\n    best_actions = counts_to_binary_action_plan(\n        counts_block=fixed_counts,\n        num_od=num_od,\n        total_step=total_step,\n        block_steps=block_steps,\n    )\n\n    replay_seed = seed if eval_seed is None else eval_seed\n    final_reward, final_traj = run_episode_with_plan(\n        build_env_fn=build_env_fn,\n        worker_idx=worker_idx,\n        action_plan=best_actions,\n        seed=replay_seed,\n    )\n\n    return best_actions, fixed_counts, final_reward, final_traj, block_logs\n\nimport time\nfrom trial_timing import reset_trial_times, append_trial_time\nos.makedirs(Config.RESULT_DIR, exist_ok=True)\n\n_trial_result_dir = RESULT_DIR if \"RESULT_DIR\" in globals() else Config.RESULT_DIR\n\n_trial_t0 = time.perf_counter()\ntrial_idx = trial\nseed_opt = int(100 + (trial_idx + 1))\n# Estimate seed (101-105) is kept separate from reproduce seeds (0-4).\nseed_eval = trial\n\nbest_actions, best_counts, final_reward, trajectory, block_logs = sequential_bayes_optimize_counts_300s(\n    build_env_fn=build_env,\n    worker_idx=trial,\n    num_od=Config.NUM_OD,\n    total_step=Config.TOTAL_STEP,\n    input_interval=Config.INPUT_INTERVAL,\n    detector_interval=Config.DETECTOR_INTERVAL,\n    init_points=0,\n    n_iter=300,\n    seed=seed_opt,\n    eval_seed=seed_eval,\n    verbose=2,\n)\n\nprint(f\"[trial {trial}] final_reward = {final_reward}\")\nprint(f\"[trial {trial}] best_actions.shape = {best_actions.shape}\")\nprint(f\"[trial {trial}] best_counts.shape = {best_counts.shape}\")\n\nwith open(os.path.join(Config.RESULT_DIR, f\"trajectory_{trial}.json\"), \"w\", encoding=\"utf-8\") as f:\n    json.dump(\n        trajectory, f, ensure_ascii=False, indent=2,\n        default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o\n    )\n\nwith open(os.path.join(Config.RESULT_DIR, f\"block_logs_{trial}.json\"), \"w\", encoding=\"utf-8\") as f:\n    json.dump(block_logs, f, ensure_ascii=False, indent=2)\n\n_elapsed_path = os.environ.get('DODE_TRIAL_ELAPSED_PATH')\nif _elapsed_path:\n    with open(_elapsed_path, 'w', encoding='utf-8') as f:\n        json.dump({'trial': int(trial), 'elapsed_sec': float(time.perf_counter() - _trial_t0)}, f)\n"
_scripts_dir = Path(Config.RL_ROOT).resolve() / "scripts"

def _run_trial_process(trial, runner_path, tmp_dir):
    elapsed_path = Path(tmp_dir) / f"elapsed_{trial}.json"
    env = os.environ.copy()
    env["DODE_SCRIPTS_DIR"] = str(_scripts_dir)
    env["DODE_TRIAL_ELAPSED_PATH"] = str(elapsed_path)
    env["PYTHONUNBUFFERED"] = "1"
    cmd = [sys.executable, str(runner_path), str(trial)]
    started = time.perf_counter()
    print(f"[trial {trial}] started")
    proc = subprocess.run(cmd, cwd=str(_scripts_dir), env=env)
    elapsed = time.perf_counter() - started
    if proc.returncode != 0:
        raise RuntimeError(f"trial {trial} failed with exit code {proc.returncode}")
    if elapsed_path.exists():
        with open(elapsed_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        elapsed = float(payload.get("elapsed_sec", elapsed))
    return int(trial), float(elapsed)

with tempfile.TemporaryDirectory(prefix="dode_main_parallel_") as _tmp_dir:
    _runner_path = Path(_tmp_dir) / "trial_runner.py"
    _runner_path.write_text(_runner_source, encoding="utf-8")

    _results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_run_trial_process, trial, _runner_path, _tmp_dir): trial for trial in TRIALS}
        for future in as_completed(futures):
            trial, elapsed = future.result()
            print(f"[trial {trial}] completed in {elapsed:.3f} sec")
            _results.append((trial, elapsed))

for trial, elapsed in sorted(_results):
    append_trial_time(_trial_result_dir, trial, elapsed)

In [ ]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
